In [ ]:
# Ollama doesn't ship Phi-3.5-MoE, so this demo runs the same agent prompt on Phi-3.5-mini
import os

from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# Ollama's OpenAI-compatible endpoint (set OLLAMA_BASE_URL in .env)
client = OpenAI(
    base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
    api_key="ollama",  # required by the client library, ignored by Ollama
)

# Phi-3.5-mini-instruct (pull it with `ollama pull phi3.5`)
model_name = "phi3.5"


In [ ]:
generation_args = {
    "max_tokens": 512,
    "temperature": 0.3,
}


In [12]:
sys_msg = """You are a helpful AI assistant, you are an agent capable of using a variety of tools to answer a question. Here are a few of the tools available to you:

- Blog: This tool helps you describe a certain knowledge point and content, and finally write it into Twitter or Facebook style content
- Translate: This is a tool that helps you translate into any language, using plain language as required

To use these tools you must always respond in JSON format containing `"tool_name"` and `"input"` key-value pairs. For example, to answer the question, "Build Muliti Agents with MOE models" you must use the calculator tool like so:

```json

{
    "tool_name": "Blog",
    "input": "Build Muliti Agents with MOE models"
}

```

Or to translate the question "can you introduce yourself in Chinese" you must respond:

```json

{
    "tool_name": "Search",
    "input": "can you introduce yourself in Chinese"
}

```

Remember just output the final result, output in JSON format containing `"agentid"`,`"tool_name"` , `"input"` and `"output"`  key-value pairs .:

```json

[


{   "agentid": "step1",
    "tool_name": "Blog",
    "input": "Build Muliti Agents with MOE models",
    "output": "........."
},

{   "agentid": "step2",
    "tool_name": "Search",
    "input": "can you introduce yourself in Chinese",
    "output": "........."
},
{
    "agentid": "final"
    "tool_name": "Result",
    "output": "........."
}
]

```

The users answer is as follows.
"""

In [ ]:
def instruction_format(sys_message: str, query: str):
    # Ollama applies Phi's <|system|> / <|user|> chat template for us
    return [
        {"role": "system", "content": sys_message},
        {"role": "user", "content": query},
    ]


In [14]:
query ='Write something about Generative AI with MOE , translate it to Chinese'

In [ ]:
input_prompt = instruction_format(sys_msg, query)


In [ ]:
input_prompt

In [ ]:
output = client.chat.completions.create(model=model_name, messages=input_prompt, **generation_args)

In [ ]:
output.choices[0].message.content